In [1]:
from blackjack import Game
from tqdm import tqdm
from collections import defaultdict
import random
import numpy as np

In [2]:
rng = np.random.default_rng(42)
random.seed(42)

In [3]:
def get_state(game: Game):
    player_card_ranks = []
    for card in game.player_hand:
        player_card_ranks.append(min(card.value, 10))

    state = (
        tuple(sorted(player_card_ranks)),
        min(game.dealer_hand[0].value, 10)
    )

    return state

In [12]:
# Q(s, 0) is stay and Q(s, 1) is hit.
ACTIONS = np.array(["S", "H"])
q_table = defaultdict(lambda: np.zeros(len(ACTIONS), dtype=np.float64))
num_times_visited = defaultdict(lambda: np.zeros(len(ACTIONS), dtype=np.int64))

n_iter = 10_000_000
epsilon = 1.0
min_epsilon = 0.01
epsilon_decay = (min_epsilon / epsilon) ** (1 / n_iter)
gamma = 0.99

In [13]:
training_outcomes = np.zeros(3, dtype=np.int64)  # losses, draws, wins

for _ in tqdm(range(n_iter)):
    game = Game(five_card_charlie=True)

    while not game.is_over:
        state = get_state(game)

        if rng.random() < epsilon:
            action_index = int(rng.integers(len(ACTIONS)))
        else:
            values = q_table[state]
            best_actions = np.flatnonzero(values == values.max())
            action_index = int(rng.choice(best_actions))

        game.take_action(ACTIONS[action_index])

        if game.is_over:
            target_q_value = game.score
        else:
            next_state = get_state(game)
            
            # current reward is 0, so bellman's equation simplifies to gamma * max Q(s', a')
            target_q_value = gamma * np.max(q_table[next_state])

        current = q_table[state][action_index]
        
        num_times_visited[state][action_index] += 1
        alpha = 1.0 / num_times_visited[state][action_index]  # schedule learning rate based on number of times a (state, action) pair is visited
        
        q_table[state][action_index] += alpha * (target_q_value - current)

    training_outcomes[game.score + 1] += 1
    epsilon = max(epsilon * epsilon_decay, min_epsilon)

print(f"Learned {len(q_table):,} states.")
print(dict(zip(("losses", "draws", "wins"), training_outcomes)))

100%|██████████| 10000000/10000000 [09:29<00:00, 17561.81it/s]

Learned 5,024 states.
{'losses': np.int64(5093587), 'draws': np.int64(711754), 'wins': np.int64(4194659)}


In [14]:
def choose_greedy_action(game: Game) -> str:
    """Choose a learned action, breaking ties randomly."""
    values = q_table.get(get_state(game))
    if values is None:
        return str(rng.choice(ACTIONS))

    return str(ACTIONS[int(np.argmax(values))])


def evaluate_agent(n_games: int = 10_000) -> dict[str, float]:
    outcomes = np.zeros(3, dtype=np.int64)

    for _ in range(n_games):
        game = Game(five_card_charlie=True)
        while not game.is_over:
            game.take_action(choose_greedy_action(game))
        outcomes[game.score + 1] += 1

    return {
        "loss_rate": outcomes[0] / n_games,
        "draw_rate": outcomes[1] / n_games,
        "win_rate": outcomes[2] / n_games,
        "average_reward": (outcomes[2] - outcomes[0]) / n_games,
    }


evaluate_agent()

{'loss_rate': np.float64(0.4746),
 'draw_rate': np.float64(0.0818),
 'win_rate': np.float64(0.4436),
 'average_reward': np.float64(-0.031)}

In [22]:
import pickle

with open('bj_q_table.pkl', 'wb') as f:
    pickle.dump(dict(q_table), f)

In [29]:
from itertools import combinations_with_replacement
import pandas as pd


def starting_hand_value(cards: tuple[int, int]) -> tuple[int, bool]:
    total = sum(cards)
    usable_ace = 1 in cards and total + 10 <= 21
    return total + 10 if usable_ace else total, usable_ace


def standard_strategy(cards: tuple[int, int], dealer_upcard: int) -> str:
    """
    The standard blackjack strategy (assuming hit and stand as the only available actions)
    """
    total, is_soft = starting_hand_value(cards)

    if is_soft:
        if total <= 17:
            return "Hit"
        if total == 18:
            return "Stand" if dealer_upcard in range(2, 9) else "Hit"
        return "Stand"

    if total <= 11:
        return "Hit"
    if total == 12:
        return "Stand" if dealer_upcard in range(4, 7) else "Hit"
    if total <= 16:
        return "Stand" if dealer_upcard in range(2, 7) else "Hit"
    return "Stand"


rank_label = {1: "A", **{rank: str(rank) for rank in range(2, 11)}}
comparison_rows = []

for cards in combinations_with_replacement(range(1, 11), 2):
    total, is_soft = starting_hand_value(cards)
    for dealer_upcard in range(1, 11):
        state = (cards, dealer_upcard)
        stand_q, hit_q = q_table.get(state, np.zeros(2))

        if stand_q > hit_q:
            learned_action = "Stand"
        elif hit_q > stand_q:
            learned_action = "Hit"
        else:
            learned_action = "Tie"

        expected_action = standard_strategy(cards, dealer_upcard)
        comparison_rows.append({
            "Player hand": f"{rank_label[cards[0]]},{rank_label[cards[1]]}",
            "Total": total,
            "Soft": is_soft,
            "Dealer": rank_label[dealer_upcard],
            "Q(Stand)": stand_q,
            "Q(Hit)": hit_q,
            "Learned": learned_action,
            "Standard": expected_action,
            "Matches": learned_action == expected_action,
        })

strategy_comparison = pd.DataFrame(comparison_rows)
print(f"Number of disagreeing states: {strategy_comparison.loc[~(strategy_comparison["Matches"]), 'Matches'].count()}")

strategy_comparison.sort_values(["Matches", "Soft", "Total", "Player hand", "Dealer"]).head(10)

Number of disagreeing states: 8


,Player hand,Total,Soft,Dealer,Q(Stand),Q(Hit),Learned,Standard,Matches
183,"2,10",12,False,4,-0.207028,-0.177326,Hit,Stand,False
185,"2,10",12,False,6,-0.158204,-0.149952,Hit,Stand,False
315,"4,8",12,False,6,-0.204259,-0.165910,Hit,Stand,False
362,"5,7",12,False,3,-0.210276,-0.224090,Stand,Hit,False
403,"6,6",12,False,4,-0.192308,-0.181668,Hit,Stand,False
405,"6,6",12,False,6,-0.226891,-0.219285,Hit,Stand,False
261,"3,10",13,False,2,-0.320941,-0.289519,Hit,Stand,False
321,"4,9",13,False,2,-0.303435,-0.280257,Hit,Stand,False
109,"2,2",4,False,10,-0.566929,-0.046175,Hit,Hit,True
101,"2,2",4,False,2,-0.333333,0.034653,Hit,Hit,True
